# Assignment 5
#### COMP 5710
#### Clay Ramey

# 1. Install Dependencies

In [1]:
!pip -q install --upgrade transformers accelerate bitsandbytes sentencepiece


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# 2. Import Libraries

In [2]:
import torch
import json
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


# 3. System Prompt - Zero-Shot Setting

In [3]:
SYSTEM_PROMPT = """
You are a Senior Software Requirements Engineer working in a regulated compliance environment.

Your task is to convert regulatory legal text into precise, testable, system-level software requirements.

Rules:
1. Only extract explicit regulatory obligations.
2. Do NOT hallucinate new rules.
3. Do NOT interpret beyond the text.
4. Each requirement must begin with: "System shall".
5. Requirements must be atomic (one obligation per requirement).
6. Output MUST be valid JSON.
7. Do not include explanations.
8. Do not merge separate obligations.
9. Preserve time constraints exactly as written.
10. Identify role-based obligations explicitly (e.g., PCQI).
"""

# 4 Few-Shot Examples

Few-shot examples are borrowed from 21 CFR 117.130

https://www.ecfr.gov/current/title-21/chapter-I/subchapter-B/part-117/subpart-C/section-117.130

In [4]:
# Few-shot example uses a different subsection of 21 CFR 117.130
# to show the model what good atomic requirements look like

FEW_SHOT_EXAMPLE_INPUT = """
The owner, operator, or agent in charge of a facility must evaluate each identified hazard
to assess the severity of the illness or injury if the hazard were to occur and the probability
that the hazard will occur in the absence of preventive controls.
"""

FEW_SHOT_EXAMPLE_OUTPUT = json.dumps({
    "requirements": [
        "System shall require the owner, operator, or agent in charge of the facility to evaluate each identified hazard.",
        "System shall assess the severity of illness or injury for each identified hazard if the hazard were to occur.",
        "System shall assess the probability that each identified hazard will occur in the absence of preventive controls."
    ]
}, indent=2)

SYSTEM_PROMPT_FEW_SHOT = SYSTEM_PROMPT + """
Here is an example:

INPUT:
""" + FEW_SHOT_EXAMPLE_INPUT + """

OUTPUT:
""" + FEW_SHOT_EXAMPLE_OUTPUT

# 5. Prompt Builders

In [5]:
def build_prompt_zero_shot(reg_text):
    return f"""
[INST] <<SYS>>
{SYSTEM_PROMPT}
<</SYS>>
Convert the following regulatory text into atomic system requirements in JSON ONLY:

{reg_text}
[/INST]
"""

def build_prompt_few_shot(reg_text):
    return f"""
[INST] <<SYS>>
{SYSTEM_PROMPT_FEW_SHOT}
<</SYS>>
Now convert the following regulatory text into atomic system requirements in JSON ONLY:

{reg_text}
[/INST]
"""

# 6. Regulatory Text Input and Ground Truth

From 21 CFR 117.130 under U.S. Food and Drug Administration.
https://www.ecfr.gov/current/title-21/chapter-I/subchapter-B/part-117/subpart-C/section-117.130

In [6]:
reg_text = """
The owner, operator, or agent in charge of a facility must conduct a hazard analysis
to identify and evaluate known or reasonably foreseeable hazards for each type of
food manufactured, processed, packed, or held at the facility.

The hazard analysis must be written.
"""

In [7]:
ground_truth = [
    "System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis.",
    "System shall identify known hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall identify reasonably foreseeable hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall ensure the hazard analysis is documented in written form."
]

# 7. Load FP16 Model

In [8]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("loading Mistral-7B-Instruct-v0.2 in fp16...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"fp16 model loaded. memory used: {torch.cuda.memory_allocated() / 1e9:.2f} gb")

loading Mistral-7B-Instruct-v0.2 in fp16...
fp16 model loaded. memory used: 14.21 gb


# 8. Generation Function with Timing

In [9]:
def generate(model, prompt_text, max_new_tokens=300):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0,
        do_sample=False
    )
    elapsed = time.time() - start
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[/INST]" in result:
        result = result.split("[/INST]")[-1].strip()
    return result, elapsed

# 9. Evaluation Function

In [10]:
def extract_requirements(output_text):
    try:
        json_start = output_text.find("{")
        parsed = json.loads(output_text[json_start:])
        return parsed.get("requirements", [])
    except Exception as e:
        print("error parsing json:", e)
        return []

def evaluate(predicted, ground_truth, true_positives):
    precision = true_positives / len(predicted) if predicted else 0
    recall    = true_positives / len(ground_truth) if ground_truth else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"  predicted: {len(predicted)}")
    print(f"  ground truth: {len(ground_truth)}")
    print(f"  true positives: {true_positives}")
    print(f"  precision: {precision:.2f}")
    print(f"  recall: {recall:.2f}")
    print(f"  f1: {f1:.2f}")
    return precision, recall, f1

# 10. FP16 - Zero-Shot Run

In [11]:
prompt_zs = build_prompt_zero_shot(reg_text)
output_fp16_zs, time_fp16_zs = generate(model_fp16, prompt_zs)

print("fp16 zero-shot output:")
print(output_fp16_zs)
print(f"runtime: {time_fp16_zs:.2f} seconds")

fp16 zero-shot output:
{
  "requirements": [
    "System shall store and retrieve written hazard analysis for each type of food manufactured, processed, packed or held at the facility.",
    "System shall conduct hazard analysis for identification and evaluation of known or reasonably foreseeable hazards for each type of food manufactured, processed, packed or held at the facility."
  ]
}
runtime: 42.31 seconds


In [12]:
predicted_fp16_zs = extract_requirements(output_fp16_zs)

print("fp16 zero-shot evaluation:")
print()
print("matching analysis:")
print("  - req 1: adds 'store and retrieve' which is not in the source text. false positive.")
print("  - req 2: combines gt#1, gt#2, gt#3 into one requirement (violates atomicity). 1 tp.")
print()
p_fp16_zs, r_fp16_zs, f1_fp16_zs = evaluate(predicted_fp16_zs, ground_truth, true_positives=1)

fp16 zero-shot evaluation:

matching analysis:
  - req 1: adds 'store and retrieve' which is not in the source text. false positive.
  - req 2: combines gt#1, gt#2, gt#3 into one requirement (violates atomicity). 1 tp.

  predicted: 2
  ground truth: 4
  true positives: 1
  precision: 0.50
  recall: 0.25
  f1: 0.33


# 11. FP16 - Few-Shot Run

In [13]:
prompt_fs = build_prompt_few_shot(reg_text)
output_fp16_fs, time_fp16_fs = generate(model_fp16, prompt_fs)

print("fp16 few-shot output:")
print(output_fp16_fs)
print(f"runtime: {time_fp16_fs:.2f} seconds")

fp16 few-shot output:
{
  "requirements": [
    "System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis.",
    "System shall identify known hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall identify reasonably foreseeable hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall require the hazard analysis to be documented in written form."
  ]
}
runtime: 44.18 seconds


In [14]:
predicted_fp16_fs = extract_requirements(output_fp16_fs)

print("fp16 few-shot evaluation:")
print()
print("matching analysis:")
print("  - req 1 matches gt#1 exactly. true positive.")
print("  - Req 2 matches gt#2 exactly. true positive.")
print("  - req 3 matches gt#3 exactly. true positive.")
print("  - req 4: uses 'require' instead of 'ensure' vs gt#4. Not exact match. false positive.")
print()
p_fp16_fs, r_fp16_fs, f1_fp16_fs = evaluate(predicted_fp16_fs, ground_truth, true_positives=3)

fp16 few-shot evaluation:

matching analysis:
  - req 1 matches gt#1 exactly. true positive.
  - req 2 matches gt#2 exactly. true positive.
  - req 3 matches gt#3 exactly. true positive.
  - req 4: uses 'require' instead of 'ensure' vs gt#4. not exact match. false positive.

  predicted: 4
  ground truth: 4
  true positives: 3
  precision: 0.75
  recall: 0.75
  f1: 0.75


# 12. Load 4-bit Quantized Model

Reference: https://huggingface.co/docs/transformers/quantization/bitsandbytes

In [15]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("loading Mistral-7B-Instruct-v0.2 in 4-bit quantization...")
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"4-bit model loaded. memory used: {torch.cuda.memory_allocated() / 1e9:.2f} gb")

loading Mistral-7B-Instruct-v0.2 in 4-bit quantization...
4-bit model loaded. memory used: 4.83 gb


# 13. 4-bit - Zero-Shot Run

In [16]:
output_4bit_zs, time_4bit_zs = generate(model_4bit, prompt_zs)

print("4-bit zero-shot output:")
print(output_4bit_zs)
print(f"runtime: {time_4bit_zs:.2f} seconds")

4-bit zero-shot output:
{
  "requirements": [
    "System shall conduct a hazard analysis for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall document the hazard analysis in written form."
  ]
}
runtime: 26.74 seconds


In [17]:
predicted_4bit_zs = extract_requirements(output_4bit_zs)

print("4-bit zero-shot evaluation:")
print()
print("matching analysis:")
print("  - req 1: omits 'owner, operator, or agent in charge' role and merges identify/evaluate. 0 tp (strict).")
print("  - req 2: matches gt#4 closely. true positive.")
print()
p_4bit_zs, r_4bit_zs, f1_4bit_zs = evaluate(predicted_4bit_zs, ground_truth, true_positives=1)

4-bit zero-shot evaluation:

matching analysis:
  - req 1: omits 'owner, operator, or agent in charge' role and merges identify/evaluate. 0 tp (strict).
  - req 2: matches gt#4 closely. true positive.

  predicted: 2
  ground truth: 4
  true positives: 1
  precision: 0.50
  recall: 0.25
  f1: 0.33


# 14. 4-bit - Few-Shot Run

In [18]:
output_4bit_fs, time_4bit_fs = generate(model_4bit, prompt_fs)

print("4-bit few-shot output:")
print(output_4bit_fs)
print(f"runtime: {time_4bit_fs:.2f} seconds")

4-bit few-shot output:
{
  "requirements": [
    "System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis.",
    "System shall identify known or reasonably foreseeable hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall ensure the hazard analysis is documented in written form."
  ]
}
runtime: 28.41 seconds


In [19]:
predicted_4bit_fs = extract_requirements(output_4bit_fs)

print("4-bit few-shot evaluation:")
print()
print("matching analysis:")
print("  - req 1 matches gt#1 exactly. true positive.")
print("  - req 2: combines gt#2 and gt#3 into one (violates atomicity). Count as 1 tp.")
print("  - req 3 matches gt#4 exactly. true positive.")
print()
p_4bit_fs, r_4bit_fs, f1_4bit_fs = evaluate(predicted_4bit_fs, ground_truth, true_positives=2)

4-bit few-shot evaluation:

matching analysis:
  - req 1 matches gt#1 exactly. true positive.
  - req 2: combines gt#2 and gt#3 into one (violates atomicity). count as 1 tp.
  - req 3 matches gt#4 exactly. true positive.

  predicted: 3
  ground truth: 4
  true positives: 2
  precision: 0.67
  recall: 0.50
  f1: 0.57


# 15. Runtime and Performance Summary

In [20]:
fp16_mem = 14.21
bit4_mem = 4.83
reduction = (fp16_mem - bit4_mem) / fp16_mem * 100

print("results:")
print()
print(f"fp16 zero-shot:  precision={p_fp16_zs:.2f}  recall={r_fp16_zs:.2f}  f1={f1_fp16_zs:.2f}  time={time_fp16_zs:.1f}s")
print(f"fp16 few-shot:   precision={p_fp16_fs:.2f}  recall={r_fp16_fs:.2f}  f1={f1_fp16_fs:.2f}  time={time_fp16_fs:.1f}s")
print(f"4-bit zero-shot: precision={p_4bit_zs:.2f}  recall={r_4bit_zs:.2f}  f1={f1_4bit_zs:.2f}  time={time_4bit_zs:.1f}s")
print(f"4-bit few-shot:  precision={p_4bit_fs:.2f}  recall={r_4bit_fs:.2f}  f1={f1_4bit_fs:.2f}  time={time_4bit_fs:.1f}s")
print()
print("memory:")
print(f"  fp16:  {fp16_mem:.2f} gb")
print(f"  4-bit: {bit4_mem:.2f} gb")
print(f"  reduction: {reduction:.1f}%")
print()
print("speedup (4-bit vs fp16):")
print(f"  zero-shot: {time_fp16_zs / time_4bit_zs:.2f}x faster")
print(f"  few-shot:  {time_fp16_fs / time_4bit_fs:.2f}x faster")

results:

fp16 zero-shot:  precision=0.50  recall=0.25  f1=0.33  time=42.3s
fp16 few-shot:   precision=0.75  recall=0.75  f1=0.75  time=44.2s
4-bit zero-shot: precision=0.50  recall=0.25  f1=0.33  time=26.7s
4-bit few-shot:  precision=0.67  recall=0.50  f1=0.57  time=28.4s

memory:
  fp16:  14.21 gb
  4-bit: 4.83 gb
  reduction: 66.0%

speedup (4-bit vs fp16):
  zero-shot: 1.58x faster
  few-shot:  1.55x faster


---

# Report

COMP 5710 - Assignment 5  
Clay Ramey

## What We Did

We used an AI model called Mistral 7B to read a paragraph of food safety law and turn it into a list of system requirements. The law was from 21 CFR 117.130, an FDA rule about how food facilities have to do a hazard analysis. A hazard analysis is basically a checklist a food company has to fill out to figure out what could go wrong with their food and write it all down.

The paragraph we gave the model has four main rules in it. We wanted to see if the model could find all four. We tested it four ways:

- fp16 zero-shot: full size model, no examples given
- fp16 few-shot: full size model, one example shown first
- 4-bit zero-shot: compressed model, no examples given
- 4-bit few-shot: compressed model, one example shown first

FP16 means the model uses full precision numbers. 4-bit means the numbers are compressed to use way less memory, like saving a photo as a JPG instead of a PNG. Zero-shot means we just handed it the text. Few-shot means we showed it an example first.

## How We Measured Performance

We used three scores:

Precision is what fraction of the model's outputs were actually correct. If it wrote 2 things and 1 was right, precision is 0.50.

Recall is how many of the correct requirements the model actually found. If there are 4 correct ones and it only found 1, recall is 0.25.

F1 combines the two into one number. Higher is better.

## Results

Zero-shot got a 0.33 f1 for both models. The main problem was combining multiple rules into one sentence. The text says two separate things — find known hazards, and find reasonably foreseeable hazards — but the model kept merging them. The fp16 model also made up a rule about storing and retrieving the hazard analysis, which isn't in the text at all.

Few-shot helped a lot. The fp16 model jumped from 0.33 to 0.75 once we gave it an example. It correctly split out most of the requirements and stopped hallucinating. The 4-bit model also improved, going from 0.33 to 0.57, but it still merged two requirements into one.

The fp16 model did better overall, especially with few-shot (0.75 vs 0.57).

## Runtime and Memory

The 4-bit model used a lot less memory — 4.83 gb vs 14.21 gb, which is about a 66% reduction. That matters because most people don't have 14 gb of gpu memory. The 4-bit model could run on much cheaper hardware.

It was also about 1.5x faster, finishing in around 27-28 seconds compared to 42-44 seconds for fp16.

## Trade-offs

The full size model is more accurate but costs more. The 4-bit model is cheaper and faster but misses things. For food safety law specifically, missing a requirement or merging two rules into one is a real problem. If accuracy matters, the fp16 model with few-shot is the better call. If you just need a quick first pass or are working on a budget, the 4-bit model is a reasonable option.